In [1]:
import boto3

session = boto3.Session(profile_name="ATLAS")
s3 = session.client("s3")
BUCKET = "atlas-acm-research-f26"

response = s3.list_objects_v2(Bucket=BUCKET)
for obj in response.get("Contents", []):
    print(obj["Key"])
print("connected!")

artifacts/baseline/
artifacts/multimodal/
dataset/
satellite/processed/
satellite/raw/
satellite/raw/1059037050.png
satellite/raw/temp/0052227492.png
satellite/raw/temp/0169170979.png
satellite/raw/temp/0209073657.png
satellite/raw/temp/0239032142.png
satellite/raw/temp/0283051477.png
satellite/raw/temp/0283051485.png
satellite/raw/temp/0299929061.png
satellite/raw/temp/0337127832.png
satellite/raw/temp/0435817145.png
satellite/raw/temp/0610827836.png
satellite/raw/temp/0673532692.png
satellite/raw/temp/0696985482.png
satellite/raw/temp/0701923200.png
satellite/raw/temp/0751881301.png
satellite/raw/temp/0762116506.png
satellite/raw/temp/0781370600.png
satellite/raw/temp/0830885774.png
satellite/raw/temp/0878795698.png
satellite/raw/temp/0944024849.png
satellite/raw/temp/0977653854.png
satellite/raw/temp/1073720218.png
satellite/raw/temp/1173822721.png
satellite/raw/temp/1227550179.png
usgs/processed/
usgs/raw/
connected!


In [3]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="dataset/")
for obj in response.get("Contents", []):
    print(obj["Key"])

dataset/
dataset/im3_baseline.csv


In [4]:
s3.upload_file("Data/eia_generator_features.csv", BUCKET, "features/eia_generator.csv")
s3.upload_file("Data/eia_rto_features.csv",       BUCKET, "features/eia_rto_region.csv")
print("uploaded!")

uploaded!


In [5]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="features/")
for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes")

features/eia_generator.csv 97864 bytes
features/eia_rto_region.csv 271417 bytes


In [7]:
import pandas as pd
s3.download_file(BUCKET, "dataset/im3_baseline.csv", "im3_baseline.csv")
pd.read_csv("im3_baseline.csv", dtype=str).columns.tolist()

['id',
 'state',
 'state_abb',
 'state_id',
 'county',
 'county_id',
 'operator',
 'ref',
 'name',
 'sqft',
 'lon',
 'lat',
 'type']

In [8]:
for f in ["Data/eia_generator_features.csv", "Data/eia_rto_features.csv"]:
    df = pd.read_csv(f, dtype=str)
    print(f, "->", df.columns.tolist()[:5], "| first id:", df["id"].iloc[0] if "id" in df.columns else "NO id COLUMN")

base = pd.read_csv("im3_baseline.csv", dtype=str)
print("baseline first id:", base["id"].iloc[0])

Data/eia_generator_features.csv -> ['id', 'nearest_plant_km', 'plants_50km', 'capacity_50km_mw', 'renewable_50km_mw'] | first id: 0002744301
Data/eia_rto_features.csv -> ['id', 'ba_code', 'ba_peak_mw', 'ba_avg_mw', 'ba_peak_to_avg'] | first id: 0002744301
baseline first id: 0002744301


In [9]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="features/")
for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes")

features/eia_generator.csv 97864 bytes
features/eia_rto_region.csv 271417 bytes


In [10]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="dataset/")
for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes")

dataset/ 0 bytes
dataset/im3_baseline.csv 146728 bytes
dataset/im3_multimodal_atlas_fueltype.csv 354188 bytes
dataset/im3_multimodal_atlas_retail.csv 132979 bytes


In [11]:
for path in ["dataset/im3_multimodal_atlas_retail.csv", "dataset/im3_multimodal_atlas_fueltype.csv"]:
    local = path.split("/")[-1]
    s3.download_file(BUCKET, path, local)
    df = pd.read_csv(local, dtype=str)
    new_cols = [c for c in df.columns if c not in base.columns]
    matched = df["id"].isin(base["id"]).sum() if "id" in df.columns else "NO id COLUMN"
    print(f"{local}: {matched} of {len(df)} ids match | new columns: {new_cols}")

im3_multimodal_atlas_retail.csv: 328 of 1147 ids match | new columns: ['size_class', 'period', 'comm_elec_price_cents_kwh', 'comm_elec_sales_mwh', 'est_mwh_per_hour', 'est_hourly_energy_cost_usd']
im3_multimodal_atlas_fueltype.csv: 1147 of 1147 ids match | new columns: ['comm_elec_price_cents_kwh', 'comm_elec_sales_mwh', 'st_share_coal', 'st_share_gas', 'st_share_nuclear', 'st_share_hydro', 'st_share_solar', 'st_share_wind', 'st_share_fossil', 'st_share_renewable', 'st_share_clean', 'st_total_gen_mwh', 'est_mwh_per_hour', 'est_hourly_energy_cost_usd']


In [12]:
retail = pd.read_csv("im3_multimodal_atlas_retail.csv", dtype=str)
bad = retail[~retail["id"].isin(base["id"])]
print(bad["id"].head(10).tolist())
print(base["id"].head(10).tolist())

['2744301', '7805491', '9474864', '13924557', '14593270', '14930068', '14997588', '15884451', '16282459', '18518989']
['0002744301', '0007805491', '0009474864', '0013924557', '0014593270', '0014930068', '0014997588', '0015884451', '0016282459', '0018518989']


In [13]:
print("baseline id lengths:", base["id"].str.len().unique())
retail["id"] = retail["id"].str.zfill(10)
print("retail match after fix:", retail["id"].isin(base["id"]).sum(), "of", len(retail))

baseline id lengths: [10]
retail match after fix: 1147 of 1147


In [ ]:
KEY = "id"

FEATURE_FILES = [
    "features/eia_generator.csv",
    "features/eia_rto_region.csv",
    "dataset/im3_multimodal_atlas_fueltype.csv",  
    "dataset/im3_multimodal_atlas_retail.csv",
]

base = pd.read_csv("im3_baseline.csv", dtype={KEY: str})
base_rows = len(base)
assert base[KEY].is_unique
print("Baseline:", base.shape)

for path in FEATURE_FILES:
    local = path.split("/")[-1]
    s3.download_file(BUCKET, path, local)
    df = pd.read_csv(local, dtype={KEY: str})
    df[KEY] = df[KEY].str.zfill(10)          # fix the zeroes for the matcha
    new_cols = [c for c in df.columns if c not in base.columns]
    matched = df[KEY].isin(base[KEY]).sum()
    print(f"{local}: {matched}/{len(df)} ids match, adding {new_cols}")
    if not df[KEY].is_unique:
        print("  WARNING: duplicate ids, keeping first")
        df = df.drop_duplicates(subset=KEY)
    base = base.merge(df[[KEY] + new_cols], on=KEY, how="left")

assert len(base) == base_rows, "Row count changed"
print("Final:", base.shape)
print(base.isna().sum()[base.isna().sum() > 0])

Baseline: (1147, 13)
eia_generator.csv: 1147/1147 ids match, adding ['nearest_plant_km', 'plants_50km', 'capacity_50km_mw', 'renewable_50km_mw', 'renewable_share_50km', 'nearest_big_plant_km']
eia_rto_region.csv: 1147/1147 ids match, adding ['ba_code', 'ba_peak_mw', 'ba_avg_mw', 'ba_peak_to_avg', 'ba_demand_coverage', 'ba_share_coal', 'ba_share_gas', 'ba_share_nuclear', 'ba_share_hydro', 'ba_share_solar', 'ba_share_wind', 'ba_share_fossil', 'ba_share_renewable', 'ba_share_clean', 'ba_total_gen_mwh']
im3_multimodal_atlas_fueltype.csv: 1147/1147 ids match, adding ['comm_elec_price_cents_kwh', 'comm_elec_sales_mwh', 'st_share_coal', 'st_share_gas', 'st_share_nuclear', 'st_share_hydro', 'st_share_solar', 'st_share_wind', 'st_share_fossil', 'st_share_renewable', 'st_share_clean', 'st_total_gen_mwh', 'est_mwh_per_hour', 'est_hourly_energy_cost_usd']
im3_multimodal_atlas_retail.csv: 1147/1147 ids match, adding ['size_class', 'period']
Final: (1147, 50)
operator    411
ref         834
name    

In [22]:
base.to_csv("im3_eia_combined.csv", index=False)
s3.upload_file("im3_eia_combined.csv", BUCKET, "dataset/im3_eia_combined.csv")
print("done!")

done!


In [23]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="dataset/")
for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"], "bytes")

dataset/ 0 bytes
dataset/im3_baseline.csv 146728 bytes
dataset/im3_eia_combined.csv 736731 bytes
dataset/im3_multimodal_atlas_fueltype.csv 354188 bytes
dataset/im3_multimodal_atlas_retail.csv 132979 bytes


In [ ]:
s3.download_file(BUCKET, "dataset/im3_eia_combined.csv", "check_combined.csv")
c = pd.read_csv("check_combined.csv", dtype={"id": str})

print("1. Shape:", c.shape, "(want 1147 rows, 50 cols)")
print("2. Unique ids:", c["id"].is_unique, "| all 10 digits:", (c["id"].str.len() == 10).all())
print("3. Any _x/_y duplicate columns:", [col for col in c.columns if col.endswith(("_x", "_y"))])
print("4. Duplicate column names:", c.columns[c.columns.duplicated()].tolist())


sources = ["eia_generator.csv", "eia_rto_region.csv",
           "im3_multimodal_atlas_retail.csv", "im3_multimodal_atlas_fueltype.csv"]
for f in sources:
    src = pd.read_csv(f, dtype={"id": str})
    src["id"] = src["id"].str.zfill(10)
    src = src.drop_duplicates(subset="id").set_index("id")
    comb = c.set_index("id")
    shared = [col for col in src.columns if col in comb.columns]
    mismatched = []
    for col in shared:
        a = src[col].reindex(comb.index)
        b = comb[col]
        both_nan = a.isna() & b.isna()
        try:
            same = ((a.astype(float) - b.astype(float)).abs() < 1e-9) | both_nan
        except (ValueError, TypeError):
            same = (a.astype(str) == b.astype(str)) | both_nan
        if not same.all():
            mismatched.append(f"{col} ({(~same).sum()} rows)")
    print(f"5. {f}: {len(shared)} columns checked, mismatches: {mismatched or 'none'}")

print("6. Rows missing retail data:")
print(c[c["comm_elec_price_cents_kwh"].isna()][["id", "state", "county"]])

1. Shape: (1147, 50) (want 1147 rows, 50 cols)
2. Unique ids: True | all 10 digits: True
3. Any _x/_y duplicate columns: []
4. Duplicate column names: []
5. eia_generator.csv: 6 columns checked, mismatches: none
5. eia_rto_region.csv: 15 columns checked, mismatches: none
5. im3_multimodal_atlas_retail.csv: 11 columns checked, mismatches: ['state_abb (3 rows)', 'comm_elec_price_cents_kwh (3 rows)', 'comm_elec_sales_mwh (3 rows)', 'est_hourly_energy_cost_usd (3 rows)']
5. im3_multimodal_atlas_fueltype.csv: 20 columns checked, mismatches: none
6. Rows missing retail data:
Empty DataFrame
Columns: [id, state, county]
Index: []


In [25]:
ret = pd.read_csv("im3_multimodal_atlas_retail.csv", dtype={"id": str})
fuel = pd.read_csv("im3_multimodal_atlas_fueltype.csv", dtype={"id": str})
ret["id"] = ret["id"].str.zfill(10)
fuel["id"] = fuel["id"].str.zfill(10)

cmp = (c[["id", "county", "state_abb", "comm_elec_price_cents_kwh"]]
       .merge(ret[["id", "state_abb"]].rename(columns={"state_abb": "retail_state_abb"}), on="id")
       .merge(fuel[["id", "comm_elec_price_cents_kwh"]].rename(columns={"comm_elec_price_cents_kwh": "fuel_price"}), on="id"))

diff = cmp[(cmp["state_abb"].astype(str) != cmp["retail_state_abb"].astype(str)) |
           (cmp["comm_elec_price_cents_kwh"].astype(str) != cmp["fuel_price"].astype(str))]
print(diff.to_string())

             id                county state_abb  comm_elec_price_cents_kwh retail_state_abb  fuel_price
35   0046128119       Hamilton County        OH                      14.06               KY       14.06
93   0192307107  San Francisco County        CA                      30.56              NaN       30.56
374  0460085343  San Francisco County        CA                      30.56              NaN       30.56


In [26]:
ids = ["0046128119", "0192307107", "0460085343"]
print(c[c["id"].isin(ids)][["id", "state_abb", "comm_elec_price_cents_kwh", "period", "comm_elec_sales_mwh", "est_hourly_energy_cost_usd"]].to_string())

             id state_abb  comm_elec_price_cents_kwh   period  comm_elec_sales_mwh  est_hourly_energy_cost_usd
35   0046128119        OH                      14.06  2026-07           6324.99033                   415.24804
93   0192307107        CA                      30.56      NaN          10778.89224                  1057.34544
374  0460085343        CA                      30.56      NaN          10778.89224                  1201.55808


In [27]:
print(c["period"].value_counts(dropna=False))

period
2026-07    1145
NaN           2
Name: count, dtype: int64


In [28]:
base["period"] = base["period"].fillna("2026-07")
base.to_csv("im3_eia_combined.csv", index=False)
s3.upload_file("im3_eia_combined.csv", BUCKET, "dataset/im3_eia_combined.csv")
print("done!")

done!


In [29]:
s3.download_file(BUCKET, "dataset/im3_eia_combined.csv", "check_combined.csv")
print(pd.read_csv("check_combined.csv")["period"].value_counts(dropna=False))

period
2026-07    1147
Name: count, dtype: int64


In [ ]:
c = pd.read_csv("im3_eia_combined.csv", dtype={"id": str})
pd.set_option("display.max_rows", 100)

eia_cols = c.columns[13:]                   
num = c[eia_cols].select_dtypes("number")
print(num.describe().T[["min", "mean", "max"]])

neg = (num < 0).sum()
print("\nColumns with negative values:", neg[neg > 0].to_dict() or "none")
print("Missing values:", c[eia_cols].isna().sum()[lambda s: s > 0].to_dict() or "none")

                                     min          mean           max
nearest_plant_km            9.526222e-03  7.094129e+00  3.466966e+01
plants_50km                 2.000000e+00  5.149172e+01  2.960000e+02
capacity_50km_mw            1.290000e+01  3.900859e+03  2.082670e+04
renewable_50km_mw           0.000000e+00  6.296142e+02  7.070800e+03
renewable_share_50km        0.000000e+00  2.169064e-01  1.000000e+00
nearest_big_plant_km        9.969323e-02  1.570193e+01  8.533200e+01
ba_peak_mw                  5.090000e+02  8.574197e+04  1.605600e+05
ba_avg_mw                   2.263543e+02  5.199836e+04  9.624162e+04
ba_peak_to_avg              1.368609e+00  1.686041e+00  2.373022e+00
ba_demand_coverage          9.972603e-01  9.999023e-01  1.000000e+00
ba_share_coal               0.000000e+00  1.505555e-01  7.897857e-01
ba_share_gas                0.000000e+00  3.714429e-01  8.642235e-01
ba_share_nuclear            0.000000e+00  1.840363e-01  5.380215e-01
ba_share_hydro              0.0000

In [31]:
checks = {
    "peak < avg demand":                   c["ba_peak_mw"] < c["ba_avg_mw"],
    "peak_to_avg < 1":                      c["ba_peak_to_avg"] < 1,
    "renewable > total capacity":           c["renewable_50km_mw"] > c["capacity_50km_mw"],
    "0 plants in 50km but capacity > 0":    (c["plants_50km"] == 0) & (c["capacity_50km_mw"] > 0),
    "nearest > 50km but plants_50km > 0":   (c["nearest_plant_km"] > 50) & (c["plants_50km"] > 0),
    "nearest < 50km but plants_50km == 0":  (c["nearest_plant_km"] < 50) & (c["plants_50km"] == 0),
}
share_cols = [col for col in c.columns if col.startswith("st_share_")]
if share_cols:
    checks["fuel shares don't sum to ~1"] = (c[share_cols].sum(axis=1) - 1).abs() > 0.02

for name, mask in checks.items():
    print(f"{name}: {int(mask.sum())} rows")

peak < avg demand: 0 rows
peak_to_avg < 1: 0 rows
renewable > total capacity: 0 rows
0 plants in 50km but capacity > 0: 0 rows
nearest > 50km but plants_50km > 0: 0 rows
nearest < 50km but plants_50km == 0: 0 rows
fuel shares don't sum to ~1: 1147 rows


In [32]:
share_cols = [col for col in c.columns if col.startswith("st_share_")]
print(share_cols)

singles = [col for col in share_cols if not col.endswith(("fossil", "renewable", "clean"))]
s = c[singles].sum(axis=1)
print("individual fuel sum: min", round(s.min(), 3), "max", round(s.max(), 3))

['st_share_coal', 'st_share_gas', 'st_share_nuclear', 'st_share_hydro', 'st_share_solar', 'st_share_wind', 'st_share_fossil', 'st_share_renewable', 'st_share_clean']
individual fuel sum: min 0.853 max 1.001


In [33]:
combo = c.groupby(["ba_code", "state_abb"]).size().reset_index(name="n")
print(combo.sort_values(["ba_code", "n"], ascending=[True, False]).to_string(index=False))
print("\nMissing ba_code:", c["ba_code"].isna().sum())

ba_code state_abb   n
    AVA        WA   4
   AZPS        AZ  20
   BANC        CA   6
   BPAT        OR  50
   BPAT        WA   4
   CHPD        WA   3
   CISO        CA  72
   CPLE        NC   3
   DOPD        WA  13
    DUK        NC  16
    EPE        NM   1
   ERCO        TX  89
   FMPP        FL   1
    FPC        FL   2
    FPL        FL   3
   GCPD        WA  38
    GVL        FL   1
   ISNE        CT   4
   ISNE        MA   2
   ISNE        NH   2
   ISNE        ME   1
    JEA        FL   1
   LDWP        CA   5
   LGEE        KY   2
   MISO        IA  56
   MISO        MI   7
   MISO        WI   6
   MISO        MN   5
   MISO        MO   3
   MISO        AR   2
   MISO        IN   2
   MISO        IL   1
   MISO        KY   1
   MISO        MS   1
   MISO        SD   1
   NEVP        NV  32
   NWMT        MT   1
   NYIS        NY  12
   PACE        UT  10
   PACE        ID   2
   PACW        OR  14
    PGE        OR  27
    PJM        VA 288
    PJM        OH  55
    PJM   

In [34]:
print(combo.iloc[25:50].to_string(index=False))

ba_code state_abb   n
   MISO        IA  56
   MISO        IL   1
   MISO        IN   2
   MISO        KY   1
   MISO        MI   7
   MISO        MN   5
   MISO        MO   3
   MISO        MS   1
   MISO        SD   1
   MISO        WI   6
   NEVP        NV  32
   NWMT        MT   1
   NYIS        NY  12
   PACE        ID   2
   PACE        UT  10
   PACW        OR  14
    PGE        OR  27
    PJM        IL  35
    PJM        MD   7
    PJM        NJ  45
    PJM        OH  55
    PJM        PA   7
    PJM        VA 288
    PNM        NM  10
   PSCO        CO  10


In [35]:
print(combo.iloc[[24]].to_string(index=False))
print(combo.iloc[50:].to_string(index=False))

ba_code state_abb  n
   MISO        AR  2
ba_code state_abb  n
   PSEI        WA 19
     SC        SC 11
   SCEG        SC  1
   SOCO        AL  3
   SOCO        GA 30
    SRP        AZ 20
   SWPP        KS  6
   SWPP        MO  6
   SWPP        NE 20
   SWPP        OK  2
    TEC        FL  2
    TVA        AL  9
    TVA        TN 15
   WACM        CO  6
   WACM        WY 14


In [37]:
base = base.rename(columns={"comm_elec_sales_mwh": "comm_elec_sales_gwh"})
base.to_csv("im3_eia_combined.csv", index=False)
s3.upload_file("im3_eia_combined.csv", BUCKET, "dataset/im3_eia_combined.csv")
print("done!", "comm_elec_sales_gwh" in base.columns)

done! True


In [1]:
import boto3, pandas as pd, geopandas as gpd

s3 = boto3.Session(profile_name="ATLAS").client("s3")
BUCKET = "atlas-acm-research-f26"
s3.download_file(BUCKET, "dataset/im3_eia_combined.csv", "check_combined.csv")
c = pd.read_csv("check_combined.csv", dtype={"id": str})

counties = gpd.read_file("Data/cb_2023_us_county_500k.zip")
pts = gpd.GeoDataFrame(c[["id", "state_abb", "county", "lat", "lon"]],
                       geometry=gpd.points_from_xy(c["lon"], c["lat"]),
                       crs="EPSG:4326").to_crs(counties.crs)
j = gpd.sjoin(pts, counties[["STUSPS", "NAMELSAD", "geometry"]], how="left", predicate="within")

no_match = j[j["STUSPS"].isna()]
wrong_state = j[j["STUSPS"].notna() & (j["STUSPS"] != j["state_abb"])]
wrong_county = j[(j["STUSPS"] == j["state_abb"]) & (j["NAMELSAD"] != j["county"])]

print("Points outside every county (bad coords?):", len(no_match))
print("State mismatches:", len(wrong_state))
print(wrong_state[["id", "state_abb", "STUSPS", "county", "NAMELSAD"]].head(20).to_string(index=False))
print("\nCounty name mismatches:", len(wrong_county))
print(wrong_county[["id", "county", "NAMELSAD"]].head(20).to_string(index=False))

Points outside every county (bad coords?): 0
State mismatches: 0
Empty DataFrame
Columns: [id, state_abb, STUSPS, county, NAMELSAD]
Index: []

County name mismatches: 1
        id         county      NAMELSAD
0830885779 Madison County Warren County


In [4]:
print("gwh column present:", "comm_elec_sales_gwh" in c.columns)
print("old mwh column gone:", "comm_elec_sales_mwh" not in c.columns)
print("shape:", c.shape)

gwh column present: True
old mwh column gone: True
shape: (1147, 50)
